# 🛒 BasketGPT — Training Notebook

Train a **lightweight causal transformer** on Instacart order sequences.

**Architecture:**
- Mini-GPT with **RoPE** (Rotary Position Embeddings)
- 2 Transformer layers, 4 attention heads, 64d embeddings
- ~3.3M parameters, weight-tied output
- Trained with **next-token prediction** on product ID sequences

**How to use:**
1. Upload `order_products__prior.csv` and `products.csv` to Colab
2. Run all cells
3. Download the 3 output files from `outputs/` directory
4. Place them in `ml_backend/data/basket/` on your local machine

In [ ]:
# ============================================================
# Cell 1: Setup & Upload Data
# ============================================================
from google.colab import files
import os

os.makedirs('outputs', exist_ok=True)

print('📁 Upload order_products__prior.csv and products.csv')
print('   (You can also mount Google Drive if files are there)')
print()

# Option A: Upload directly
# uploaded = files.upload()

# Option B: Mount Google Drive (uncomment if using Drive)
from google.colab import drive
drive.mount('/content/drive')

# Set paths — EDIT THESE if your files are in a different location
DRIVE_PATH = '/content/drive/MyDrive/Dataset'  # Change this to your path

ORDER_PRODUCTS_PATH = os.path.join(DRIVE_PATH, 'order_products__prior.csv')
PRODUCTS_PATH = os.path.join(DRIVE_PATH, 'products.csv')

# Verify files exist
for p in [ORDER_PRODUCTS_PATH, PRODUCTS_PATH]:
    if os.path.exists(p):
        print(f'✅ Found: {p}')
    else:
        print(f'❌ NOT FOUND: {p}')
        print('   Please update the path above')

In [ ]:
# ============================================================
# Cell 2: Load & Prepare Data
# ============================================================
import pandas as pd
import numpy as np
import pickle
from collections import Counter
import time

print('📊 Loading datasets...')
t0 = time.time()

# Load order-product mapping
df = pd.read_csv(ORDER_PRODUCTS_PATH)
print(f'   order_products__prior: {len(df):,} rows')

# Load product names
products_df = pd.read_csv(PRODUCTS_PATH)
product_lookup = dict(zip(products_df['product_id'], products_df['product_name']))
print(f'   products: {len(products_df):,} products')

# Sort by order_id and add_to_cart_order to preserve sequence
df = df.sort_values(['order_id', 'add_to_cart_order'])

# Group by order_id to create baskets
print('\n🧺 Building baskets...')
baskets = df.groupby('order_id')['product_id'].apply(list).values
print(f'   Total baskets: {len(baskets):,}')

# Filter: keep baskets with >= 3 items
baskets = [b for b in baskets if len(b) >= 3]
print(f'   Baskets with >= 3 items: {len(baskets):,}')

# Dataset statistics
sizes = [len(b) for b in baskets]
print(f'\n📈 Basket statistics:')
print(f'   Mean size: {np.mean(sizes):.1f}')
print(f'   Median size: {np.median(sizes):.1f}')
print(f'   95th percentile: {np.percentile(sizes, 95):.0f}')
print(f'   99th percentile: {np.percentile(sizes, 99):.0f}')

# Find unique product IDs
all_product_ids = set()
for b in baskets:
    all_product_ids.update(b)
max_product_id = max(all_product_ids)
print(f'\n   Unique products: {len(all_product_ids):,}')
print(f'   Max product ID: {max_product_id}')

elapsed = time.time() - t0
print(f'\n⏱️  Data loading took {elapsed:.1f}s')

In [ ]:
# ============================================================
# Cell 3: Save Product Lookup
# ============================================================
with open('outputs/product_lookup.pkl', 'wb') as f:
    pickle.dump(product_lookup, f)
print(f'✅ Saved product_lookup.pkl ({len(product_lookup):,} products)')

In [ ]:
# ============================================================
# Cell 4: Model Definition — BasketGPT with RoPE
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

print(f'🔧 PyTorch version: {torch.__version__}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🖥️  Device: {device}')
if device.type == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


# ---------- RoPE ----------

def precompute_rope_freqs(dim, max_seq_len, theta=10000.0):
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
    t = torch.arange(max_seq_len).float()
    freqs = torch.outer(t, freqs)
    return freqs.cos(), freqs.sin()


def apply_rope(x, cos, sin):
    seq_len = x.shape[2]
    head_dim = x.shape[3]
    cos = cos[:seq_len].unsqueeze(0).unsqueeze(0)
    sin = sin[:seq_len].unsqueeze(0).unsqueeze(0)
    x1 = x[..., :head_dim // 2]
    x2 = x[..., head_dim // 2:]
    return torch.cat([x1 * cos - x2 * sin, x2 * cos + x1 * sin], dim=-1)


# ---------- Attention with RoPE ----------

class RoPEAttention(nn.Module):
    def __init__(self, embed_dim, n_heads, dropout=0.1, max_seq_len=50):
        super().__init__()
        assert embed_dim % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = embed_dim // n_heads
        self.embed_dim = embed_dim

        self.q_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.k_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.v_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.out_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        cos, sin = precompute_rope_freqs(self.head_dim, max_seq_len)
        self.register_buffer('rope_cos', cos)
        self.register_buffer('rope_sin', sin)
        mask = torch.triu(torch.ones(max_seq_len, max_seq_len), diagonal=1).bool()
        self.register_buffer('causal_mask', mask)

    def forward(self, x):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        q = apply_rope(q, self.rope_cos, self.rope_sin)
        k = apply_rope(k, self.rope_cos, self.rope_sin)
        scale = math.sqrt(self.head_dim)
        attn = (q @ k.transpose(-2, -1)) / scale
        attn = attn.masked_fill(self.causal_mask[:T, :T].unsqueeze(0).unsqueeze(0), float('-inf'))
        attn = F.softmax(attn, dim=-1)
        attn = self.attn_dropout(attn)
        out = (attn @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(out))


# ---------- Transformer Block ----------

class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, n_heads, ffn_dim, dropout=0.1, max_seq_len=50):
        super().__init__()
        self.ln1 = nn.LayerNorm(embed_dim)
        self.attn = RoPEAttention(embed_dim, n_heads, dropout, max_seq_len)
        self.ln2 = nn.LayerNorm(embed_dim)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, ffn_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(ffn_dim, embed_dim), nn.Dropout(dropout),
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x


# ---------- BasketGPT ----------

class BasketGPT(nn.Module):
    PAD_TOKEN = 0
    BOS_TOKEN = 49689
    UNK_TOKEN = 49690

    def __init__(self, vocab_size=49691, embed_dim=64, n_heads=4,
                 n_layers=2, ffn_dim=256, max_seq_len=50, dropout=0.1):
        super().__init__()
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.n_heads = n_heads
        self.n_layers = n_layers
        self.ffn_dim = ffn_dim
        self.max_seq_len = max_seq_len
        self.dropout_rate = dropout

        self.token_embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=self.PAD_TOKEN)
        self.embed_dropout = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, n_heads, ffn_dim, dropout, max_seq_len)
            for _ in range(n_layers)
        ])
        self.ln_final = nn.LayerNorm(embed_dim)
        self.output_head = nn.Linear(embed_dim, vocab_size, bias=False)
        self.output_head.weight = self.token_embedding.weight  # Weight tying
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.padding_idx is not None:
                module.weight.data[module.padding_idx].zero_()

    def forward(self, input_ids):
        x = self.embed_dropout(self.token_embedding(input_ids))
        for block in self.blocks:
            x = block(x)
        return self.output_head(self.ln_final(x))

    def get_config(self):
        return {
            'vocab_size': self.vocab_size, 'embed_dim': self.embed_dim,
            'n_heads': self.n_heads, 'n_layers': self.n_layers,
            'ffn_dim': self.ffn_dim, 'max_seq_len': self.max_seq_len,
            'dropout': self.dropout_rate, 'pad_token': self.PAD_TOKEN,
            'bos_token': self.BOS_TOKEN, 'unk_token': self.UNK_TOKEN,
        }


# ---------- Instantiate ----------

# Dataset-specific hyperparameters
VOCAB_SIZE = 49691      # 49688 products + PAD(0) + BOS(49689) + UNK(49690)
EMBED_DIM = 64          # Embedding dimension
N_HEADS = 4             # Attention heads
N_LAYERS = 2            # Transformer layers
FFN_DIM = 256           # Feed-forward hidden dim
MAX_SEQ_LEN = 50        # Covers 99th percentile basket + BOS + room
DROPOUT = 0.1

model = BasketGPT(
    vocab_size=VOCAB_SIZE, embed_dim=EMBED_DIM, n_heads=N_HEADS,
    n_layers=N_LAYERS, ffn_dim=FFN_DIM, max_seq_len=MAX_SEQ_LEN,
    dropout=DROPOUT
).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\n🏗️  BasketGPT initialized:')
print(f'   Total params: {total_params:,}')
print(f'   Trainable:    {trainable:,}')
print(f'   Model size:   ~{total_params * 4 / 1e6:.1f} MB (float32)')
print(f'   Architecture: {N_LAYERS}L / {N_HEADS}H / {EMBED_DIM}D / FFN={FFN_DIM}')
print(f'   Position:     RoPE (θ=10000)')
print(f'   Weight tying: Yes (output ↔ embedding)')

In [ ]:
# ============================================================
# Cell 5: Dataset & DataLoader (Memory Efficient)
# ============================================================
from torch.utils.data import Dataset, DataLoader
import random

BOS_TOKEN = 49689
PAD_TOKEN = 0

class BasketDataset(Dataset):
    def __init__(self, baskets, max_seq_len=50, augment=True):
        self.baskets = baskets
        self.max_seq_len = max_seq_len
        self.augment = augment

    def __len__(self):
        return len(self.baskets)

    def __getitem__(self, idx):
        basket = list(self.baskets[idx])
        if self.augment:
            random.shuffle(basket)
        sequence = [BOS_TOKEN] + basket
        if len(sequence) > self.max_seq_len:
            sequence = sequence[:self.max_seq_len]
        input_seq = sequence[:-1]
        target_seq = sequence[1:]
        pad_len = (self.max_seq_len - 1) - len(input_seq)
        input_seq = input_seq + [PAD_TOKEN] * pad_len
        target_seq = target_seq + [PAD_TOKEN] * pad_len
        return (torch.tensor(input_seq, dtype=torch.long), torch.tensor(target_seq, dtype=torch.long))

random.seed(42)
shuffled = list(baskets)
random.shuffle(shuffled)
split_idx = int(0.95 * len(shuffled))
train_baskets = shuffled[:split_idx]
val_baskets = shuffled[split_idx:]

train_dataset = BasketDataset(train_baskets, max_seq_len=MAX_SEQ_LEN, augment=True)
val_dataset = BasketDataset(val_baskets, max_seq_len=MAX_SEQ_LEN, augment=False)

# Micro-batch size for memory efficiency
BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'፦ Dataset ready (Micro-Batch Size: {BATCH_SIZE})')

In [ ]:
# ============================================================
# Cell 6: Training Loop (Memory Optimized Gradient Accumulation)
# ============================================================
import torch
import math
import time
import gc
from torch.amp import autocast, GradScaler

# Clear VRAM
gc.collect()
torch.cuda.empty_cache()

EPOCHS = 5
LR = 1e-3
WEIGHT_DECAY = 0.01
LABEL_SMOOTHING = 0.1
WARMUP_STEPS = 500
LOG_INTERVAL = 200
ACCUMULATION_STEPS = 8  # Process 32 at a time, update every 8 steps (Effective Batch: 256)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_TOKEN, label_smoothing=LABEL_SMOOTHING)
scaler = GradScaler('cuda')

total_steps = EPOCHS * (len(train_loader) // ACCUMULATION_STEPS)

def get_lr(step):
    if step < WARMUP_STEPS: return LR * step / WARMUP_STEPS
    progress = min(1.0, (step - WARMUP_STEPS) / max(1, total_steps - WARMUP_STEPS))
    return LR * 0.5 * (1.0 + math.cos(math.pi * progress))

def train_epoch(epoch):
    model.train()
    total_loss, total_correct, total_tokens = 0, 0, 0
    optimizer.zero_grad()

    for batch_idx, (input_ids, targets) in enumerate(train_loader):
        input_ids, targets = input_ids.to(device), targets.to(device)

        with autocast('cuda'):
            logits = model(input_ids)
            loss = criterion(logits.view(-1, model.vocab_size), targets.view(-1))
            loss = loss / ACCUMULATION_STEPS

        scaler.scale(loss).backward()

        if (batch_idx + 1) % ACCUMULATION_STEPS == 0:
            global_step = (epoch * len(train_loader) // ACCUMULATION_STEPS) + (batch_idx // ACCUMULATION_STEPS)
            lr = get_lr(global_step)
            for param_group in optimizer.param_groups: param_group['lr'] = lr

            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            # Periodic memory cleanup
            if (batch_idx + 1) % (ACCUMULATION_STEPS * 10) == 0:
                gc.collect()
                torch.cuda.empty_cache()

        total_loss += loss.item() * ACCUMULATION_STEPS
        with torch.no_grad():
            preds = logits.argmax(dim=-1)
            mask = targets != PAD_TOKEN
            total_correct += ((preds == targets) & mask).sum().item()
            total_tokens += mask.sum().item()

        if (batch_idx + 1) % LOG_INTERVAL == 0:
            print(f'  Batch {batch_idx+1:,}/{len(train_loader):,} | Loss: {total_loss/(batch_idx+1):.4f} | Acc: {total_correct/total_tokens*100:.2f}%')

    return total_loss / len(train_loader), total_correct / total_tokens * 100

@torch.no_grad()
def validate():
    model.eval()
    total_loss, total_correct, total_tokens = 0, 0, 0
    for input_ids, targets in val_loader:
        input_ids, targets = input_ids.to(device), targets.to(device)
        with autocast('cuda'):
            logits = model(input_ids)
            loss = criterion(logits.view(-1, model.vocab_size), targets.view(-1))
        total_loss += loss.item()
        preds = logits.argmax(dim=-1)
        mask = targets != PAD_TOKEN
        total_correct += ((preds == targets) & mask).sum().item()
        total_tokens += mask.sum().item()
    return total_loss / len(val_loader), total_correct / total_tokens * 100

best_val_loss = float('inf')
history = []
for epoch in range(EPOCHS):
    t0 = time.time()
    train_loss, train_acc = train_epoch(epoch)
    val_loss, val_acc = validate()
    elapsed = time.time() - t0
    history.append({'epoch': epoch + 1, 'train_loss': train_loss, 'train_acc': train_acc, 'val_loss': val_loss, 'val_acc': val_acc})
    print(f'\n፡ Epoch {epoch+1}/{EPOCHS} ({elapsed:.0f}s): Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%')
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "outputs/basket_gpt.pt")

In [ ]:
# ============================================================
# Cell 7: Training Curves
# ============================================================
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs = [h['epoch'] for h in history]

ax1.plot(epochs, [h['train_loss'] for h in history], 'b-o', label='Train Loss')
ax1.plot(epochs, [h['val_loss'] for h in history], 'r-o', label='Val Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.set_title('Loss Curves')
ax1.grid(True, alpha=0.3)

ax2.plot(epochs, [h['train_acc'] for h in history], 'b-o', label='Train Acc')
ax2.plot(epochs, [h['val_acc'] for h in history], 'r-o', label='Val Acc')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.legend()
ax2.set_title('Accuracy Curves')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/training_curves.png', dpi=150)
plt.show()
print('📈 Training curves saved')

In [ ]:
# ============================================================
# Cell 8: Save Config & Test Inference
# ============================================================
import json

# Save config
config = model.get_config()
with open('outputs/basket_gpt_config.json', 'w') as f:
    json.dump(config, f, indent=2)
print(f'✅ Config saved:')
print(json.dumps(config, indent=2))

# Load best model for testing
model.load_state_dict(torch.load('outputs/basket_gpt.pt', map_location=device))
model.eval()

print('\n' + '=' * 70)
print('🧪 Test Inference — Autoregressive Basket Completion')
print('=' * 70)

# Test with some sample carts
test_carts = [
    [24852, 13176, 21137],              # 3-item cart
    [47209, 47766, 21903, 13176],       # 4-item cart
    [196],                               # single item
]

@torch.no_grad()
def test_generate(cart_ids, num_suggestions=5, temperature=0.7, top_k=50):
    model.eval()
    BOS = model.BOS_TOKEN
    UNK = model.UNK_TOKEN
    
    sequence = [BOS] + cart_ids
    already_seen = set(cart_ids)
    suggestions = []
    
    for _ in range(num_suggestions):
        input_tensor = torch.tensor([sequence], dtype=torch.long, device=device)
        logits = model(input_tensor)
        next_logits = logits[0, -1, :].float()
        next_logits /= temperature
        
        next_logits[0] = float('-inf')   # PAD
        next_logits[BOS] = float('-inf') # BOS
        next_logits[UNK] = float('-inf') # UNK
        for pid in already_seen:
            if 0 <= pid < model.vocab_size:
                next_logits[pid] = float('-inf')
        for s in suggestions:
            next_logits[s] = float('-inf')
        
        if top_k > 0:
            topk_vals, topk_idx = torch.topk(next_logits, min(top_k, model.vocab_size))
            mask = torch.full_like(next_logits, float('-inf'))
            mask.scatter_(0, topk_idx, topk_vals)
            next_logits = mask
        
        probs = F.softmax(next_logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1).item()
        confidence = probs[next_token].item()
        
        suggestions.append(next_token)
        sequence.append(next_token)
    
    return suggestions


for cart in test_carts:
    cart_names = [product_lookup.get(pid, f'ID:{pid}') for pid in cart]
    print(f'\n🛒 Cart: {cart_names}')
    
    suggestions = test_generate(cart, num_suggestions=5, temperature=0.7)
    for i, pid in enumerate(suggestions, 1):
        name = product_lookup.get(pid, f'ID:{pid}')
        print(f'   {i}. {name} (ID: {pid})')

print('\n✅ Inference test complete!')

In [ ]:
# ============================================================
# Cell 9: Evaluation Metrics — Recall@K and NDCG@K
# ============================================================

@torch.no_grad()
def evaluate_metrics(model, val_baskets, K_values=[5, 10], num_samples=1000):
    """Evaluate basket completion quality with held-out products."""
    model.eval()
    random.seed(42)

    samples = random.sample(val_baskets, min(num_samples, len(val_baskets)))

    results = {K: {'hits': 0, 'ndcg': 0.0, 'total': 0} for K in K_values}

    for basket in samples:
        if len(basket) < 4:
            continue

        # Hold out the last product as ground truth
        ground_truth = basket[-1]
        max_K = max(K_values)
        
        # Ensure sequence length stays within model limit (MAX_SEQ_LEN)
        # seq = [BOS] + cart + suggestions. Total must be <= MAX_SEQ_LEN
        # So cart must be <= MAX_SEQ_LEN - 1 - max_K
        cart_limit = model.max_seq_len - 1 - max_K
        cart = basket[:-1][-cart_limit:]

        suggestions = test_generate(cart, num_suggestions=max_K, temperature=0.3, top_k=100)

        for K in K_values:
            top_k_preds = suggestions[:K]
            results[K]['total'] += 1

            if ground_truth in top_k_preds:
                results[K]['hits'] += 1
                rank = top_k_preds.index(ground_truth) + 1
                results[K]['ndcg'] += 1.0 / math.log2(rank + 1)

    print('\n፨ Evaluation Metrics:')
    print('-' * 50)
    for K in K_values:
        r = results[K]
        recall = r['hits'] / r['total'] * 100 if r['total'] > 0 else 0
        ndcg = r['ndcg'] / r['total'] if r['total'] > 0 else 0
        print(f'   Recall@{K}: {recall:.2f}% | NDCG@{K}: {ndcg:.4f} | Samples: {r["total"]}')
    print('-' * 50)

# Run evaluation
evaluate_metrics(model, val_baskets, K_values=[5, 10], num_samples=2000)

In [ ]:
# ============================================================
# Cell 10: Download Outputs
# ============================================================
import os

print('📥 Output files for download:')
for f in os.listdir('outputs'):
    path = os.path.join('outputs', f)
    size_mb = os.path.getsize(path) / 1e6
    print(f'   {f}: {size_mb:.1f} MB')

print('\n📋 Place these 3 files in your local project:')
print('   ml_backend/data/basket/basket_gpt.pt')
print('   ml_backend/data/basket/basket_gpt_config.json')
print('   ml_backend/data/basket/product_lookup.pkl')

# Auto-download (works in Colab)
try:
    from google.colab import files
    for f in ['basket_gpt.pt', 'basket_gpt_config.json', 'product_lookup.pkl']:
        files.download(os.path.join('outputs', f))
except:
    print('\n⚠️  Auto-download not available. Download from Files panel (left sidebar).')